<a href="https://colab.research.google.com/github/ashnaimran06/my-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashnaimran06/my-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


Rule: I will prioritize pages for refresh when they are stale and still have meaningful search visibility. A page will receive a higher score when it has not been updated for a long time and has a higher number of Google Search impressions. Pages that are not stale or do not have meaningful visibility will receive a lower score.

Reason code: STALE_VISIBLE

Action label: REFRESH

In [2]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [4]:
con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Warehouse connected")

Warehouse connected


In [10]:
#SIGNAL 1 - STALENESS

query = """
SELECT
    CASE
        WHEN date_diff('day', content_updated_date, DATE '2026-03-31') >= 180
            THEN '180+ days'
        WHEN date_diff('day', content_updated_date, DATE '2026-03-31') >= 90
            THEN '90-179 days'
        ELSE 'Under 90 days'
    END AS staleness_bucket,
    COUNT(*) AS n
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
)
WHERE content_updated_date IS NOT NULL
GROUP BY staleness_bucket
ORDER BY staleness_bucket
"""

con.sql(query)

┌──────────────────┬────────┐
│ staleness_bucket │   n    │
│     varchar      │ int64  │
├──────────────────┼────────┤
│ 180+ days        │  87853 │
│ 90-179 days      │  12987 │
│ Under 90 days    │ 418766 │
└──────────────────┴────────┘

**Verdict:** CONFIRMED — 87,853 content items are 180+ days since their last update, so staleness provides a meaningful pool of pages that could be considered for refresh.

In [11]:
#SIGNAL 2 - SEARCH VISIBILITY

query = """
SELECT
    CASE
        WHEN gsc_impressions >= 500 THEN '500+ impressions'
        WHEN gsc_impressions > 0 THEN '1-499 impressions'
        ELSE '0 impressions'
    END AS visibility_bucket,
    COUNT(*) AS n
FROM (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY client_hash_id, content_hash_id
)
GROUP BY visibility_bucket
ORDER BY visibility_bucket
"""

con.sql(query)

┌───────────────────┬────────┐
│ visibility_bucket │   n    │
│      varchar      │ int64  │
├───────────────────┼────────┤
│ 0 impressions     │ 154699 │
│ 1-499 impressions │ 114814 │
│ 500+ impressions  │  61924 │
└───────────────────┴────────┘

**Verdict:** CONFIRMED — 61,924 content items have 500+ impressions in March 2026, showing that there is a meaningful group of pages with search visibility that can be prioritized alongside staleness.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


query = """
SELECT
    c.client_hash_id,
    c.content_hash_id,
    c.content_updated_date,
    COALESCE(p.gsc_impressions, 0) AS gsc_impressions
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
) c
LEFT JOIN (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY client_hash_id, content_hash_id
) p
ON c.client_hash_id = p.client_hash_id
AND c.content_hash_id = p.content_hash_id
"""

queue = con.sql(query).df()

queue["days_since_last_update"] = (
    queue["content_updated_date"].max() - queue["content_updated_date"]
).dt.days

queue["stale"] = queue["days_since_last_update"] >= 180
queue["visible"] = queue["gsc_impressions"] >= 500

queue["score"] = (
    queue["stale"].astype(int)
    * queue["visible"].astype(int)
    * queue["gsc_impressions"]
)

queue["reason_code"] = "STALE_VISIBLE"
queue["action"] = "REFRESH"

queue = queue.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

output = queue[[
    "rank",
    "client_hash_id",
    "content_hash_id",
    "score",
    "reason_code",
    "action"
]]

output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

output.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rank,client_hash_id,content_hash_id,score,reason_code,action
0,1,client_20259bd6705d81d4,content_097459d155cccb26,37930.0,STALE_VISIBLE,REFRESH
1,2,client_20259bd6705d81d4,content_f2df5a8a9057783e,35980.0,STALE_VISIBLE,REFRESH
2,3,client_20259bd6705d81d4,content_ac4e2d9d3bbb06de,33348.0,STALE_VISIBLE,REFRESH
3,4,client_20259bd6705d81d4,content_66d1fffc91f4f029,31038.0,STALE_VISIBLE,REFRESH
4,5,client_20259bd6705d81d4,content_9598a57544925111,24066.0,STALE_VISIBLE,REFRESH
5,6,client_20259bd6705d81d4,content_b956947c822af734,22502.0,STALE_VISIBLE,REFRESH
6,7,client_20259bd6705d81d4,content_0d2aaf57d7146812,21060.0,STALE_VISIBLE,REFRESH
7,8,client_20259bd6705d81d4,content_b361694d518f80e2,18103.0,STALE_VISIBLE,REFRESH
8,9,client_20259bd6705d81d4,content_283e87bc4e224d58,16388.0,STALE_VISIBLE,REFRESH
9,10,client_20259bd6705d81d4,content_3f962469cb61a7c9,16019.0,STALE_VISIBLE,REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = queue.head(20).copy()

top20[[
    "rank",
    "client_hash_id",
    "content_hash_id",
    "days_since_last_update",
    "gsc_impressions",
    "score",
    "reason_code",
    "action"
]]

,rank,client_hash_id,content_hash_id,days_since_last_update,gsc_impressions,score,reason_code,action
0,1,client_20259bd6705d81d4,content_097459d155cccb26,221,37930.0,37930.0,STALE_VISIBLE,REFRESH
1,2,client_20259bd6705d81d4,content_f2df5a8a9057783e,221,35980.0,35980.0,STALE_VISIBLE,REFRESH
2,3,client_20259bd6705d81d4,content_ac4e2d9d3bbb06de,221,33348.0,33348.0,STALE_VISIBLE,REFRESH
3,4,client_20259bd6705d81d4,content_66d1fffc91f4f029,221,31038.0,31038.0,STALE_VISIBLE,REFRESH
4,5,client_20259bd6705d81d4,content_9598a57544925111,220,24066.0,24066.0,STALE_VISIBLE,REFRESH
5,6,client_20259bd6705d81d4,content_b956947c822af734,221,22502.0,22502.0,STALE_VISIBLE,REFRESH
6,7,client_20259bd6705d81d4,content_0d2aaf57d7146812,220,21060.0,21060.0,STALE_VISIBLE,REFRESH
7,8,client_20259bd6705d81d4,content_b361694d518f80e2,221,18103.0,18103.0,STALE_VISIBLE,REFRESH
8,9,client_20259bd6705d81d4,content_283e87bc4e224d58,221,16388.0,16388.0,STALE_VISIBLE,REFRESH
9,10,client_20259bd6705d81d4,content_3f962469cb61a7c9,221,16019.0,16019.0,STALE_VISIBLE,REFRESH


| Rank | Action  | Reason code   | Confidence note                                         | What would make it wrong                                                              |
| ---: | ------- | ------------- | ------------------------------------------------------- | ------------------------------------------------------------------------------------- |
|    1 | REFRESH | STALE_VISIBLE | High confidence: 221 days stale and 37,930 impressions. | Could be wrong if the page is intentionally evergreen or already updated elsewhere.   |
|    2 | REFRESH | STALE_VISIBLE | High confidence: 221 days stale and 35,980 impressions. | Could be wrong if the page is intentionally evergreen or does not need changes.       |
|    3 | REFRESH | STALE_VISIBLE | High confidence: 221 days stale and 33,348 impressions. | Could be wrong if the high visibility does not mean the content needs refreshing.     |
|    4 | REFRESH | STALE_VISIBLE | High confidence: 221 days stale and 31,038 impressions. | Could be wrong if the page is already performing well for a deliberate reason.        |
|    5 | REFRESH | STALE_VISIBLE | High confidence: 220 days stale and 24,066 impressions. | Could be wrong if the content is intentionally left unchanged.                        |
|    6 | REFRESH | STALE_VISIBLE | High confidence: 221 days stale and 22,502 impressions. | Could be wrong if the page is evergreen and needs no update.                          |
|    7 | REFRESH | STALE_VISIBLE | High confidence: 220 days stale and 21,060 impressions. | Could be wrong if the impressions are temporary or the page is already suitable.      |
|    8 | REFRESH | STALE_VISIBLE | High confidence: 221 days stale and 18,103 impressions. | Could be wrong if updating the content would not improve its usefulness.              |
|    9 | REFRESH | STALE_VISIBLE | High confidence: 221 days stale and 16,388 impressions. | Could be wrong if the page is intentionally maintained without frequent updates.      |
|   10 | REFRESH | STALE_VISIBLE | High confidence: 221 days stale and 16,019 impressions. | Could be wrong if the page's topic does not require fresh information.                |
|   11 | REFRESH | STALE_VISIBLE | High confidence: 221 days stale and 15,879 impressions. | Could be wrong if the page is already accurate and useful despite its age.            |
|   12 | REFRESH | STALE_VISIBLE | High confidence: 221 days stale and 14,774 impressions. | Could be wrong if staleness alone is not a good indicator of improvement opportunity. |
|   13 | REFRESH | STALE_VISIBLE | High confidence: 220 days stale and 10,024 impressions. | Could be wrong if the page is evergreen or intentionally unchanged.                   |
|   14 | REFRESH | STALE_VISIBLE | High confidence: 220 days stale and 9,961 impressions.  | Could be wrong if the page does not need a content update despite being old.          |
|   15 | REFRESH | STALE_VISIBLE | High confidence: 220 days stale and 9,737 impressions.  | Could be wrong if the page's current content is still appropriate.                    |
|   16 | REFRESH | STALE_VISIBLE | High confidence: 221 days stale and 9,177 impressions.  | Could be wrong if refreshing the page would not address its actual performance issue. |
|   17 | REFRESH | STALE_VISIBLE | High confidence: 221 days stale and 8,423 impressions.  | Could be wrong if the page is intentionally evergreen.                                |
|   18 | REFRESH | STALE_VISIBLE | High confidence: 220 days stale and 8,352 impressions.  | Could be wrong if the page's age does not indicate a real refresh opportunity.        |
|   19 | REFRESH | STALE_VISIBLE | High confidence: 221 days stale and 7,544 impressions.  | Could be wrong if the page already satisfies the user's search intent.                |
|   20 | REFRESH | STALE_VISIBLE | High confidence: 221 days stale and 6,853 impressions.  | Could be wrong if the page is accurate and does not benefit from an update.           |


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The weaker picks are the pages near the bottom of the top 20 because they have lower impression counts. They still meet the rule because they are stale and have at least 500 impressions, but the rule does not consider whether the page is actually declining or whether refreshing it would improve performance. A page could also be intentionally evergreen or already useful, so the recommendation could be wrong.

The weakest picks in the top 20 are therefore ranks 18–20. They have 8,352, 7,544, and 6,853 impressions respectively. Their main weakness is that the rule only uses staleness and visibility, so high impressions do not prove that a refresh is needed.

### Leakage check

No product flags or future windows were used in this baseline. The rule uses `content_updated_date` from the content table and March 2026 Google Search impressions from the daily performance data. I did not use the declining label, `trend_pct`, `trend_direction`, or any future performance window. The score is therefore based only on signals available at the chosen decision point.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.